# Magangin — Data Cleaning Pipeline

**Tujuan:** Membersihkan dataset hasil scraping `raw_jobs_20260429_2018magangin_jobs_20260429_2036.csv` agar siap
digunakan untuk analisis dan pengembangan fitur rekomendasi magang.

Pembersihan mencakup:
- Menghapus noise pada kolom `company_name` (pola "ulasan" dari Glints/Kalibrr)
- Mengisi nilai null `company_name` dengan ekstraksi dari `link`
- Mengisi null `location_raw` dengan string kosong
- Menambahkan flag `is_clean`

---

## Daftar Isi
1. [Load Data](#1-load-data)
2. [Assess — Before Cleaning](#2-assess--before-cleaning)
3. [Fix company_name](#3-fix-company_name)
4. [Handle Nulls](#4-handle-nulls)
5. [Validate — After Cleaning](#5-validate--after-cleaning)
6. [Save Output](#6-save-output)

## 1. Load Data

Import library yang dibutuhkan dan muat dataset dari path relatif.
Kita tampilkan shape dan 3 baris pertama untuk memastikan data berhasil dimuat.

In [6]:
import pandas as pd
import re
import numpy as np

# ---------------------------------------------------------------------------
# Load dataset
# ---------------------------------------------------------------------------
RAW_PATH = "../data/magangin_jobs_20260429_2036.csv"

df = pd.read_csv(RAW_PATH)
ORIGINAL_ROW_COUNT = len(df)   

print(f"Shape  : {df.shape}")
print(f"Kolom  : {list(df.columns)}")
print()
df.head(3)

Shape  : (48, 17)
Kolom  : ['source', 'title', 'link', 'location_raw', 'company_name', 'description_raw', 'skills', 'skills_count', 'role', 'location_city', 'region', 'jogja_tag', 'jabodetabek_tag', 'scraped_at', 'roadmap_url', 'is_tech', 'score']



,source,title,link,location_raw,company_name,description_raw,skills,skills_count,role,location_city,region,jogja_tag,jabodetabek_tag,scraped_at,roadmap_url,is_tech,score
0,jobstreet,Junior Software Engineer,https://id.jobstreet.com/id/job/91648206,Jakarta Raya,PT Technet Vision Indonesia,Lewati ke konten\nJobstreet\nCari lowongan\nCa...,"agile, angular, api, aws, cicd, css, docker, g...",18,it-general,jakarta,Jabodetabek,False,True,2026-04-29 20:29:18,https://roadmap.sh/computer-science,True,60
1,jobstreet,Software Engineer Internship,https://id.jobstreet.com/id/job/91761487,Tangerang,OPPO Indonesia,Lewati ke konten\nJobstreet\nCari lowongan\nCa...,"api, css, fastapi, git, go, java, javascript, ...",14,it-general,tangerang,Jabodetabek,False,True,2026-04-29 20:23:38,https://roadmap.sh/computer-science,True,51
2,jobstreet,Full Stack Software Engineer (CAD Rooms Cloud ...,https://id.jobstreet.com/id/job/91584877,Jakarta Raya,Cavalry Collective Pte. Ltd,Lewati ke konten\nJobstreet\nCari lowongan\nCa...,"api, aws, cicd, docker, git, go, java, javascr...",15,fullstack,jakarta,Remote,False,False,2026-04-29 20:29:58,https://roadmap.sh/full-stack,True,49


## 2. Assess — Before Cleaning

Sebelum melakukan pembersihan, kita perlu memahami kondisi aktual data:
- Kolom mana yang mengandung nilai null?
- Seberapa banyak `company_name` yang terkontaminasi pola "ulasan"?
- Contoh baris bermasalah untuk referensi visual.

In [7]:
# ---------------------------------------------------------------------------
# 2a. Missing values per kolom (hanya yang > 0)
# ---------------------------------------------------------------------------
missing = df.isnull().sum()
missing_nonzero = missing[missing > 0]

print("=" * 45)
print(" Missing Values per Kolom (hanya > 0)")
print("=" * 45)
if missing_nonzero.empty:
    print("  ✅ Tidak ada nilai null.")
else:
    print(missing_nonzero.to_string())
print()

# ---------------------------------------------------------------------------
# 2b. Jumlah company_name yang mengandung pola "ulasan"
# ---------------------------------------------------------------------------
ulasan_mask = df["company_name"].fillna("").str.contains(r"\d+\s*ulasan", case=False, regex=True)
print(f"company_name mengandung pola 'ulasan' : {ulasan_mask.sum()} baris")
print()

# ---------------------------------------------------------------------------
# 2c. Contoh baris bermasalah
# ---------------------------------------------------------------------------
print("=" * 45)
print(" Contoh: company_name noise (ulasan)")
print("=" * 45)
display(df.loc[ulasan_mask, ["company_name", "link", "source"]].head(5))

print("=" * 45)
print(" Contoh: baris dengan company_name null")
print("=" * 45)
null_company = df["company_name"].isnull()
if null_company.any():
    display(df.loc[null_company, ["company_name", "link", "source"]].head(5))
else:
    print("  Tidak ada company_name yang null.")

print("=" * 45)
print(" Contoh: baris dengan location_raw null")
print("=" * 45)
null_loc = df["location_raw"].isnull()
if null_loc.any():
    cols = [c for c in ["company_name", "location_raw", "location_city"] if c in df.columns] 
    display(df.loc[null_loc, cols].head(5))

    print("  Tidak ada location_raw yang null.")

 Missing Values per Kolom (hanya > 0)
location_raw    13
skills           4

company_name mengandung pola 'ulasan' : 0 baris

 Contoh: company_name noise (ulasan)


,company_name,link,source


 Contoh: baris dengan company_name null
  Tidak ada company_name yang null.
 Contoh: baris dengan location_raw null


,company_name,location_raw,location_city
5,Glints Taploker,NaN,jakarta
6,PT Sigma Global Teknologi,NaN,jakarta
7,GITS Indonesia,NaN,bandung
9,The Originote,NaN,jakarta
15,Meda,NaN,surabaya


  Tidak ada location_raw yang null.


## 3. Fix company_name

**Strategi pembersihan (berurutan):**

1. **Regex strip** — hapus pola `<angka> ulasan` dari string `company_name` (noise dari Glints).
2. **Strip whitespace** — bersihkan spasi sisa di awal/akhir.
3. **Fallback dari `link`** — jika masih null setelah langkah 1–2, ekstrak nama perusahaan dari URL:
   - **Kalibrr**: pola `/c/<slug>/jobs/` → slug di-title-case, `-` → spasi
   - **Glints**: pola `/companies/<slug>/` → idem
4. **Default** — jika tidak ada pola yang cocok → `"Unknown"`

In [8]:
# ---------------------------------------------------------------------------
# Simpan salinan kolom asli untuk perbandingan
# ---------------------------------------------------------------------------
original_company = df["company_name"].copy()

# ---------------------------------------------------------------------------
# 3a. Hapus pola "\d+ ulasan" (case-insensitive)
# ---------------------------------------------------------------------------
df["company_name"] = df["company_name"].fillna("").str.replace(
    r"\s*\d+\s*ulasan", "", case=False, regex=True
)


# ---------------------------------------------------------------------------
# 3b. Strip whitespace sisa
# ---------------------------------------------------------------------------
df["company_name"] = df["company_name"].str.strip()

# Ubah string kosong hasil strip menjadi NaN agar fallback bisa jalan
df["company_name"] = df["company_name"].replace("", np.nan)

# ---------------------------------------------------------------------------
# 3c. Fallback: ekstrak nama dari kolom link
# ---------------------------------------------------------------------------
def extract_company_from_link(link: str) -> str:
    """Ekstrak nama perusahaan dari URL Kalibrr atau Glints."""
    if not isinstance(link, str):
        return "Unknown"

    # Kalibrr: https://www.kalibrr.id/c/<slug>/jobs/...
    m = re.search(r"/c/([^/]+)/jobs", link)
    if m:
        return m.group(1).replace("-", " ").title()

    # Glints: https://glints.com/id/opportunities/jobs/.../companies/<slug>/...
    #         atau https://glints.com/.../companies/<slug>/
    m = re.search(r"/companies/([^/]+)/", link)
    if m:
        return m.group(1).replace("-", " ").title()

    return "Unknown"


null_mask = df["company_name"].isnull()
if null_mask.any():
    df.loc[null_mask, "company_name"] = df.loc[null_mask, "link"].apply(
        extract_company_from_link
    )

# ---------------------------------------------------------------------------
# 3d. Print before vs after untuk baris yang berubah
# ---------------------------------------------------------------------------
changed_mask = original_company.fillna("") != df["company_name"].fillna("")
changed_count = changed_mask.sum()

print(f"Jumlah baris yang berubah : {changed_count}")
print()

if changed_count > 0:
    comparison = pd.DataFrame({
        "BEFORE": original_company[changed_mask].values,
        "AFTER" : df.loc[changed_mask, "company_name"].values,
        "link"  : df.loc[changed_mask, "link"].values,
    })
    print("=" * 70)
    print(" Before vs After — company_name")
    print("=" * 70)
    display(comparison)

Jumlah baris yang berubah : 0



## 4. Handle Nulls

**Keputusan per kolom:**

| Kolom | Perlakuan | Alasan |
|---|---|---|
| `location_raw` | Isi dengan `""` (string kosong) | Diperlukan kolom string; null menyebabkan error downstream |
| `skills` | Biarkan null | Null berarti "skills tidak diketahui", bukan data error |

Selain itu, kita tambahkan kolom `is_clean = True` sebagai flag bahwa baris sudah melewati pipeline cleaning ini.

In [9]:
# ---------------------------------------------------------------------------
# 4a. location_raw null → isi dengan string kosong
# ---------------------------------------------------------------------------
before_loc_null = df["location_raw"].isnull().sum()
df["location_raw"] = df["location_raw"].fillna("")
after_loc_null = df["location_raw"].isnull().sum()

print(f"location_raw null → BEFORE: {before_loc_null}  |  AFTER: {after_loc_null}")

# ---------------------------------------------------------------------------
# 4b. skills null → dibiarkan (tidak diubah)
# ---------------------------------------------------------------------------
print(f"skills null       → TETAP : {df['skills'].isnull().sum()} (tidak diubah)")

# ---------------------------------------------------------------------------
# 4c. Tambah kolom is_clean = True
# ---------------------------------------------------------------------------
df["is_clean"] = True

print()
print(f"Kolom is_clean ditambahkan → semua baris: {df['is_clean'].all()}")
print(f"Shape setelah handle nulls : {df.shape}")

location_raw null → BEFORE: 13  |  AFTER: 0
skills null       → TETAP : 4 (tidak diubah)

Kolom is_clean ditambahkan → semua baris: True
Shape setelah handle nulls : (48, 18)


## 5. Validate — After Cleaning

Langkah validasi untuk memastikan:
- Tidak ada null baru yang muncul secara tidak sengaja
- Jumlah baris tetap **55** (tidak ada yang di-drop)
- Kolom kritis `company_name`, `location_raw`, dan `skills` berstatus sesuai ekspektasi

In [10]:
# ---------------------------------------------------------------------------
# 5a. Missing values per kolom setelah cleaning
# -----------------"----------------------------------------------------------
missing_after = df.isnull().sum()
missing_after_nonzero = missing_after[missing_after > 0]

print("=" * 45)
print(" Missing Values SETELAH Cleaning")
print("=" * 45)
if missing_after_nonzero.empty:
    print("  ✅ Tidak ada nilai null.")
else:
    print(missing_after_nonzero.to_string())
print()

# ---------------------------------------------------------------------------
# 5b. Jumlah baris sebelum vs sesudah
# ---------------------------------------------------------------------------
EXPECTED_ROWS = ORIGINAL_ROW_COUNT  
actual_rows = len(df)
row_status = "✅ OK" if actual_rows == EXPECTED_ROWS else "❌ MISMATCH"

print(f"Jumlah baris sebelum cleaning : {EXPECTED_ROWS}")
print(f"Jumlah baris sesudah cleaning : {actual_rows}  {row_status}")
print()

# ---------------------------------------------------------------------------
# 5c. Null count untuk kolom kritis
# ---------------------------------------------------------------------------
print("=" * 45)
print(" Null Count — Kolom Kritis")
print("=" * 45)
display(
    df[["company_name", "location_raw", "skills"]]
    .isnull()
    .sum()
    .rename("null_count")
    .to_frame()
)

 Missing Values SETELAH Cleaning
skills    4

Jumlah baris sebelum cleaning : 48
Jumlah baris sesudah cleaning : 48  ✅ OK

 Null Count — Kolom Kritis


,null_count
company_name,0
location_raw,0
skills,4


## 6. Save Output

Simpan dataframe yang sudah bersih ke file baru **tanpa** menimpa file asli.

- Output: `../data/magangin_jobs_cleaned.csv`
- Encoding: `utf-8-sig` (BOM) agar Excel membacanya dengan benar
- File asli `magangin_jobs_20260423_0013.csv` **tidak disentuh**

In [11]:
import os

# ---------------------------------------------------------------------------
# Path output — JANGAN overwrite file asli
# ---------------------------------------------------------------------------
OUTPUT_PATH = "../data/magangin_jobs_cleaned.csv"

# Guardrail: pastikan tidak menimpa file raw
assert OUTPUT_PATH != RAW_PATH, "Output path sama dengan file asli! Batalkan."

# Simpan
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

# Konfirmasi
abs_path = os.path.abspath(OUTPUT_PATH)
saved_rows = len(pd.read_csv(OUTPUT_PATH))

print("File berhasil disimpan!")
print(f"   Path  : {abs_path}")
print(f"   Baris : {saved_rows} baris tersimpan")
print(f"   Kolom : {list(df.columns)}")

File berhasil disimpan!
   Path  : /Users/admin/Desktop/noobies/DBS DS/magangin/data/magangin_jobs_cleaned.csv
   Baris : 48 baris tersimpan
   Kolom : ['source', 'title', 'link', 'location_raw', 'company_name', 'description_raw', 'skills', 'skills_count', 'role', 'location_city', 'region', 'jogja_tag', 'jabodetabek_tag', 'scraped_at', 'roadmap_url', 'is_tech', 'score', 'is_clean']
